# Entity links 점검
합성 CSV 검증과 합성 그래프의 증거 보존 병합을 확인합니다. DB에 연결하지 않습니다.


In [ ]:
from pathlib import Path
import csv, json, sys, tempfile
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/card_benefit').exists())
MODULE_ROOT = REPO_ROOT / 'src/card_benefit'
if str(MODULE_ROOT) not in sys.path: sys.path.insert(0, str(MODULE_ROOT))
from payment_card.file_pipeline.graph.entity_links_loader import validate_links
from payment_card.file_pipeline.chat.entity_dedupe import dedupe_subgraph
CHECK_DIR = Path(tempfile.mkdtemp(prefix='entity-links-check-'))


## 합성 링크 CSV 검증


In [ ]:
link_csv = CHECK_DIR / 'links.csv'
with link_csv.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['group_id','kind','canonical_node_id','member_node_id','same_props','distinct_pages','scope','link_type','merge_target'])
    writer.writeheader()
    writer.writerows([
        {'group_id':'G1','kind':'Benefit','canonical_node_id':'synthetic-a','member_node_id':'synthetic-a','same_props':'true','distinct_pages':1,'scope':'within_card','link_type':'SAME_AS_STRONG','merge_target':'true'},
        {'group_id':'G1','kind':'Benefit','canonical_node_id':'synthetic-a','member_node_id':'synthetic-b','same_props':'true','distinct_pages':1,'scope':'within_card','link_type':'SAME_AS_STRONG','merge_target':'true'},
        {'group_id':'G2','kind':'Benefit','canonical_node_id':'synthetic-c','member_node_id':'synthetic-c','same_props':'true','distinct_pages':2,'scope':'across_cards','link_type':'SIMILAR','merge_target':'false'},
        {'group_id':'G2','kind':'Benefit','canonical_node_id':'synthetic-c','member_node_id':'synthetic-d','same_props':'true','distinct_pages':2,'scope':'across_cards','link_type':'SIMILAR','merge_target':'false'},
    ])
report = validate_links(within_csv=link_csv)
report


## 합성 서브그래프 병합 및 근거 보존


In [ ]:
synthetic_graph = {
    'nodes': [
        {'id':'synthetic-a','name':'합성 혜택','label':'Benefit','props':{'source_text':'합성 근거 A','source_text_pages':[1]}},
        {'id':'synthetic-b','name':'합성 혜택','label':'Benefit','props':{'source_text':'합성 근거 B','source_text_pages':[1,2]}},
        {'id':'synthetic-c','name':'다른 카드 혜택','label':'Benefit','props':{}},
        {'id':'synthetic-d','name':'다른 카드 혜택','label':'Benefit','props':{}},
    ],
    'edges': [],
    'evidence': [{'node_id':'synthetic-a','page_no':1},{'node_id':'synthetic-b','page_no':2}],
    'seeds': ['synthetic-b','synthetic-d'],
}
merged = dedupe_subgraph(synthetic_graph, report['merge_mapping'])
[(node['id'], node.get('merged_member_count', 1), node.get('props', {}).get('source_text_pages')) for node in merged['nodes']], merged['evidence']
